In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
import os 
from thefuzz import process
import pandas as pd
import os
import re
import holidays
import tkinter as tk
from tkinter import filedialog
from collections import defaultdict
from clase_reportes_new import ReportClassNew
import json
rc = ReportClassNew()

In [2]:

ruta = rc.validar_ruta()
ruta_contabilidad = ruta / 'data' / 'contabilidad'
df_base = rc.consolidar_carpeta(ruta_carpeta=ruta_contabilidad / 'odoo' )

df_base = df_base.rename(columns={'Cuenta': 'Cuenta Origen'})
df_base['Cuenta'] = df_base['Cuenta Origen'].str.split(' ', regex=True).str[0]
df_base['Nombre cuenta'] = df_base['Cuenta Origen'].str.split(' ', regex=True).str[1:].apply(lambda x: ' '.join(x) if isinstance(x, list) else '')
df_base['N1'] = df_base['Cuenta Origen'].astype(str).str[0]
df_base['N2'] = df_base['Cuenta Origen'].astype(str).str[:2]
df_base['N3'] = df_base['Cuenta Origen'].astype(str).str[:4]

# Define la columna nivel
df_base['Nivel']  =np.where(df_base['N2']=='41', 'Ingreso Operativo',
        np.where(df_base['N2']=='42', 'Otros ingresos',
                np.where(df_base['N2']=='52', 'Gastos5 operacionales',
                        np.where(df_base['N2']=='53', 'Gastos No Operacionales',
                                    np.where(df_base['N2']=='61', 'Costo directo de ventas', 
                                            'Revisar'
                                    )
                        )
                )
        )
)

df_niveles =pd.read_excel(ruta_contabilidad / 'base_cuentas.xlsx', sheet_name='niveles')
df_concepto_unico =pd.read_excel(ruta_contabilidad / 'base_cuentas.xlsx', sheet_name='cuentas_concepto_uni')
df_concepto =pd.read_excel(ruta_contabilidad / 'base_cuentas.xlsx', sheet_name='concepto_depende_cc')
df_cc =pd.read_excel(ruta_contabilidad / 'base_cuentas.xlsx', sheet_name='CC')
df_concepto_doble =pd.read_excel(ruta_contabilidad / 'base_cuentas.xlsx', sheet_name='doble concepto')

influencer =pd.read_excel(ruta_contabilidad / 'base_cuentas.xlsx', sheet_name='INFLUENCER')

df_base['N3'] = df_base['N3'].astype(int)

df_base['Cuenta'] = df_base['Cuenta'].astype(int)

df_base_merge = df_base.merge(df_niveles, left_on='N3', right_on='cuenta', how='left').drop(columns='cuenta')

def extraer_clave(diccionario_str):
    if pd.isna(diccionario_str):
        return None
    try:
        diccionario = json.loads(diccionario_str)
        return list(diccionario.keys())[0]
    except Exception:
        return None

df_base_merge = df_base_merge.rename(columns={'Distribución analítica': 'Distribución analítica ori'})

df_base_merge['Distribución analítica'] = df_base_merge['Distribución analítica ori'].apply(extraer_clave)



Buscando archivos con extensión '.xlsx' en: G:\Otros ordenadores\Mi portátil\VENTA MENSUAL\data\contabilidad\odoo
  - Archivo 'Apunte contable (account.move.line) (4).xlsx' leído correctamente.
  - Archivo 'Apunte contable (account.move.line) (10).xlsx' leído correctamente.
  - Archivo 'Apunte contable (account.move.line) (13).xlsx' leído correctamente.
Concatenando todos los archivos...
¡Consolidación completada!


In [3]:
df_base_merge

,Línea de distribución de impuestos de emisor,Precisión analítica,Fecha,Número,Cuenta Origen,Creado el,NIT,Contacto,Etiqueta,Importe en divisa,...,Distribución analítica ori,Creado por,Cuenta,Nombre cuenta,N1,N2,N3,Nivel,Niveles,Distribución analítica
0,NaN,2,2026-12-29,BNK6/2026/0001,429581001 AJUSTE A MILES,2026-01-27 11:19:25,900167764,SURTICOSMETICOS HF EU,Write-Off,-0.52,...,NaN,ENRIQUEZ BEDOYA VALENTINA,429581001,AJUSTE A MILES,4,42,4295,Otros ingresos,Diversos,None
1,NaN,2,2026-02-12,FE2158,41353801 VENTA DE COSMETICOS GRAVADO 19%,2026-02-12 08:45:17,1036641715,KELLY PARRA,[PCNKIT12] KIT CRECIMIENTO ACTIVO,-173109.24,...,"{""116"": 100.0}",OdooBot,41353801,VENTA DE COSMETICOS GRAVADO 19%,4,41,4135,Ingreso Operativo,Venta de cosméticos,116
2,NaN,2,2026-02-12,FE2157,41353801 VENTA DE COSMETICOS GRAVADO 19%,2026-02-12 08:45:14,1114873791,Karen Dayana Delgado,[PCN26] SHAMPOO CRECIMIENTO Y CAIDA,-37815.13,...,"{""116"": 100.0, ""62,111,100,105"": 100.0}",OdooBot,41353801,VENTA DE COSMETICOS GRAVADO 19%,4,41,4135,Ingreso Operativo,Venta de cosméticos,116
3,NaN,2,2026-02-12,FE2156,41353801 VENTA DE COSMETICOS GRAVADO 19%,2026-02-12 08:45:12,1000618464,Danna carolina Sierra ocampo,[PCN31] SPORT ULTRA ACONDICIONADOR,-36974.79,...,"{""116"": 100.0, ""62,111,101,104"": 100.0}",OdooBot,41353801,VENTA DE COSMETICOS GRAVADO 19%,4,41,4135,Ingreso Operativo,Venta de cosméticos,116
4,NaN,2,2026-02-12,FE2156,41353801 VENTA DE COSMETICOS GRAVADO 19%,2026-02-12 08:45:12,1000618464,Danna carolina Sierra ocampo,[PCNKIT16] KIT CONTROL GRASA Y CRECIMIENTO,-137394.96,...,"{""116"": 100.0}",OdooBot,41353801,VENTA DE COSMETICOS GRAVADO 19%,4,41,4135,Ingreso Operativo,Venta de cosméticos,116
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
802299,NaN,2,2025-01-02,STJ/2025/0005,613538001 COSTO DE VENTAS COSMETICOS,2025-01-02 08:47:29,1003764437,Mariajose Castañeda Garcia,OFC/OUT/38642 - MASCARILLA ANCESTRAL PARA EL C...,6766.16,...,NaN,Camila Sanchez,613538001,COSTO DE VENTAS COSMETICOS,6,61,6135,Costo directo de ventas,Costo de ventas,None
802300,NaN,2,2025-01-02,STJ/2025/0004,613538001 COSTO DE VENTAS COSMETICOS,2025-01-02 08:47:29,1003764437,Mariajose Castañeda Garcia,OFC/OUT/38642 - DUAL CREMA PARA PEINAR,6214.27,...,NaN,Camila Sanchez,613538001,COSTO DE VENTAS COSMETICOS,6,61,6135,Costo directo de ventas,Costo de ventas,None
802301,NaN,2,2025-01-02,STJ/2025/0003,613538001 COSTO DE VENTAS COSMETICOS,2025-01-02 08:47:29,1003764437,Mariajose Castañeda Garcia,OFC/OUT/38642 - TRATAMIENTO LA POCION,5664.51,...,NaN,Camila Sanchez,613538001,COSTO DE VENTAS COSMETICOS,6,61,6135,Costo directo de ventas,Costo de ventas,None
802302,NaN,2,2025-01-02,STJ/2025/0002,613538001 COSTO DE VENTAS COSMETICOS,2025-01-02 08:47:29,1003764437,Mariajose Castañeda Garcia,OFC/OUT/38642 - DUTONIC (TONICO CAPILAR),8323.56,...,NaN,Camila Sanchez,613538001,COSTO DE VENTAS COSMETICOS,6,61,6135,Costo directo de ventas,Costo de ventas,None


In [69]:
df_base_merge['Distribución analítica ori'].apply(lambda x: print((','.join(json.loads(x).keys())) if pd.notna(x) else None))

None
116
116,62,111,100,105
116,62,111,101,104
116
62,111,112,104
62,111,61,105
62,111,61,103
62,111,61,106
62,111,61,107
None
None
116
116
116
116
116,62,111,96,104
116
62,111,114,106
62,111,82,104
116
116,62,111,101,104
116
116,62,111,61,105
116
116
116,62,111,61,107
116,62,111,100,105
116
116
116,62,111,100,105
116,62,111,61,107
116,62,111,101,104
116
116
116
116
116
116,62,111,96,104
116
116
116
116
116
116,62,111,100,105
116,62,111,83,107
116,62,111,61,103
116,62,111,61,107
116,62,111,114,106
116
116
116
116
116,62,111,114,103
116,62,111,114,107
116,62,111,100,105
116
116,62,111,61,107
116
116
116,62,111,112,104
116,62,111,100,105
116,62,111,61,106
116,62,111,100,105
116
116,62,111,82,104
116,62,111,96,104
116,62,111,96,104
116,62,111,114,103
116,62,111,114,107
116,62,111,114,105
116,62,111,101,104
116,62,111,113,104
116,62,111,101,104
116,62,111,82,104
116,62,111,112,104
116,62,111,82,104
116
116,62,111,98,103
116
116
116,62,111,61,103
116,62,111,61,107
116,62,111,61,105
116
116


KeyboardInterrupt: 

In [47]:
df_base_merge['Distribución analítica'].value_counts()

Distribución analítica
5       162368
6        11657
3         6424
116       3889
4         3823
         ...  
88           1
38           1
40           1
35           1
5,19         1
Name: count, Length: 86, dtype: int64

In [ ]:

# Ajustes manuales de asignación de centro de costo y concepto
df_base_merge['N1'] = df_base_merge['N1'].astype(str)
df_base_merge['N2'] = df_base_merge['N2'].astype(str)
df_base_merge['N3'] = df_base_merge['N3'].astype(str)

df_base_merge.loc[
    (df_base_merge['N3'] == '4135'),
    'Distribución analítica', 
] = '6'

df_base_merge.loc[
    (df_base_merge['N3'] == '4175') & 
    (df_base_merge['Diario']!="Facturas de cliente Cali"),
    'Distribución analítica', 
] = '6'

df_base_merge.loc[
    (df_base_merge['N1'] == '6') & (df_base_merge['Distribución analítica ori'].isna()),
    'Distribución analítica'
] =  '6'

df_base_merge.loc[
    (df_base_merge['N2'] == '42') & (df_base_merge['Distribución analítica ori'].isna()),
    'Distribución analítica'
] = '6'

df_base_merge.loc[(df_base_merge['Distribución analítica'].isna()) & 
            (df_base_merge['Número'].str.startswith('BNK')) &
                (df_base_merge['Cuenta Origen'].isin(['530515001 COMISIONES','530505002 GRAVAMEN CUATRO POR MIL', '530505001 CUOTA DE MANEJO']))
            , 'Distribución analítica'
            ] = '7'

df_base_merge.loc[(df_base_merge['Distribución analítica'].isna()) & 
            (df_base_merge['Número'].str.startswith('BNK')) &
                (df_base_merge['Cuenta Origen'].isin(['539595001 AJUSTE A MILES']))
            , 'Distribución analítica'
            ] = '6' 

df_base_merge.loc[(df_base_merge['Distribución analítica'].isna()) & 
            (df_base_merge['Número'].str.startswith('STJ')) &
            (~df_base_merge['Contacto'].isin(influencer['Contacto'].unique().tolist()))
            , 'Distribución analítica'
            ] = '6'  # validar si es clientre cc ==comercial  o infulerce cc== marketing ==

df_base_merge.loc[(df_base_merge['Distribución analítica'].isna()) & 
            (df_base_merge['Número'].str.startswith('STJ')) &
            (df_base_merge['Contacto'].isin(influencer['Contacto'].unique().tolist()))
            , 'Distribución analítica'
            ] = '4'  # validar si es clientre cc ==comercial  o infulerce cc== marketing ==

mask = df_base_merge['Distribución analítica'].fillna('').str.contains('5,')

df_base_merge.loc[mask, 'Distribución analítica'] = (
    df_base_merge.loc[mask, 'Distribución analítica']
    .apply(lambda x: x.split(',')[1].strip())
)

    # df_base_merge[df_base_merge['Distribución analítica']
    # .fillna('0').str.contains('5,')]

df_cc['cc'] = df_cc['cc'].astype(str)

df_base_merge = df_base_merge.merge(df_cc[['cc','Nombre Cencosto', 'ADM/VTAS','Origen' ]],
                                    left_on='Distribución analítica', right_on='cc', how='left').drop(columns='cc')

df_base_merge = df_base_merge.merge(df_concepto_unico, on='Cuenta', how='left')


df_concepto['Nombre Cencosto'] = df_concepto['Nombre Cencosto'].str.upper().str.strip()

df_base_merge['Nombre Cencosto'] = df_base_merge['Nombre Cencosto'].str.upper().str.strip()


df_base_merge = df_base_merge.merge(df_concepto, on=['Cuenta','Nombre Cencosto' ], how='left')
# Crea la columna conceto con base la los coceptos unicos y los que necesitan cc


df_base_merge['Concepto_uni'] = df_base_merge['Concepto_uni'].fillna('Sin datos')
df_base_merge['Concepto_cc'] = df_base_merge['Concepto_cc'].fillna('Sin datos')

# df_base_merge['Concepto'] = np.where(df_base_merge['Concepto_uni'].isna(), df_base_merge['Concepto_cc'], df_base_merge['Concepto_uni'])
df_base_merge = df_base_merge.reset_index(drop=True)
df_base_merge['Concepto'] = np.where(
    df_base_merge['Concepto_uni']=='Sin datos',
    df_base_merge['Concepto_cc'],
    df_base_merge['Concepto_uni']
)

df_base_merge.columns

df_concepto_doble['id'] = df_concepto_doble['Nombre Cencosto'] + df_concepto_doble['Cuenta'].astype(str)
df_concepto_doble = df_concepto_doble.drop_duplicates(subset=['id'], keep='first')
df_base_merge = df_base_merge.merge(df_concepto_doble, on=['Cuenta','Nombre Cencosto'],  how='left')


# Verifica las cuentas que no tienen concepto
df_cuentas = df_base_merge[df_base_merge['Concepto'].isna()][['Cuenta','Nombre Cencosto', 'Distribución analítica']]
df_cuentas = df_cuentas.drop_duplicates(subset=['Cuenta', 'Nombre Cencosto',], keep='first')
df_concepto = df_concepto.drop_duplicates(subset=['Cuenta', 'Nombre Cencosto'])


df_concepto_doble = df_concepto_doble.drop_duplicates(subset=['Cuenta'])

df_cuentas= df_cuentas.merge(df_concepto, on=['Cuenta','Nombre Cencosto'], how='left').merge(df_concepto_doble, on=['Cuenta','Nombre Cencosto'], how='left')
df_cuentas = df_cuentas.fillna('Sin datos')


df_cuentas['Estado Cuenta'] = np.where(
    (df_cuentas['Concepto_cc']=="Sin datos") & (df_cuentas['Concepto_doble']== 'Sin datos'),
    'Cuenta Nueva',
    np.where(
        df_cuentas['Concepto_doble'].notna(),
        'Cuenta Doble Concepto',
        'Revisar'
    )
)

df_cuentas = df_cuentas[['Cuenta', 'Nombre Cencosto','Estado Cuenta', 'Distribución analítica']]

# Elimina las columnas que no son necesarias
df_base_merge = df_base_merge.drop(columns=['Concepto_uni', 'Concepto_cc'])

# return df_base_merge, df_cuentas



In [ ]:

def archivos_contabilidad(rc):
"""
Crea y consolida los archivos procesados por odoo

"""
ruta = rc.validar_ruta()
ruta_contabilidad = ruta / 'data' / 'contabilidad'

df_base_merge, df_cuentas = rc.contabilidad()

max_date = df_base_merge['Fecha'].max()
min_date = df_base_merge['Fecha'].min()
min_date.strftime('%d-%m-%Y')
ruta_base = ruta_contabilidad / 'base' / f'base_{min_date.strftime('%d-%m-%Y')}_{max_date.strftime('%d-%m-%Y')}.csv'
df_base_merge.to_csv(ruta_base, sep=";", index=False, encoding='utf-8', decimal=',')

centros_no_re = df_base_merge[(df_base_merge['Nombre Cencosto'].isna())&
            (~df_base_merge['Distribución analítica'].isna())
            ][['Distribución analítica ori','Distribución analítica', 'Nombre Cencosto' ]].drop_duplicates()
# Centros de costo mal clasificados
cc_corregir = df_base_merge[df_base_merge['Distribución analítica ori'].fillna('').str.count(':')>1]

# Genera el archivo de los casos sin centro de costos
sin_cc = df_base_merge[df_base_merge['Distribución analítica'].isna()]
sin_cc.to_excel(ruta_contabilidad / 'sin_cc.xlsx', index=False)

# Genera el archivo con los errores
with pd.ExcelWriter(ruta_contabilidad / 'correciones.xlsx', engine='openpyxl') as writer:
    sin_cc.to_excel(writer, index=False, sheet_name='Sin CC')
    cc_corregir.to_excel(writer, index=False, sheet_name='Corregir CC')
    centros_no_re.to_excel(writer, index=False, sheet_name='CC_no_registrados')
    df_cuentas.to_excel(writer, index=False, sheet_name='Cuentas sin concep')


# Crea un archivo para cada persona de contabilidad
digitadores = sin_cc['Creado por'].unique()
ruta_errores = ruta_contabilidad / 'correcciones'
dicc = {}
for i in digitadores:
    base = sin_cc[sin_cc['Creado por']==i]
    base.to_csv(ruta_errores / f'{i}.csv', index=False, sep=';', decimal=',', encoding='utf-8')
    dicc[i] = f'{i}.csv'

df_base_consol =  rc.consolidar_carpeta(extension='csv', encoding='utf-8', sep=';', decimal=',', ruta_carpeta= ruta_contabilidad / 'base')
# pd.read_csv(r"C:\Users\Dataa\Desktop\VENTAS\VENTA MENSUAL\data\contabilidad\base\base_ene_jun_2025.csv",encoding='utf-8', sep=';')

df_base_consol = df_base_consol.loc[:, ~df_base_consol.columns.str.contains('^Unnamed')]

df_base_consol.to_csv(ruta_contabilidad / 'base_consolidada.csv', encoding='utf-8', sep=';', decimal=',', index=False)


# df_base_consol.to_excel(ruta_contabilidad / 'base_consolidada.xlsx',  index=False)

return df_base_consol, dicc



In [8]:
# from IPython.display import display, HTML
# import matplotlib.pyplot as plt
# import io
# import base64
# import humanize

# informes_por_zona = {}

# for zona in cobertura_hoy['ZONA'].to_list():
#     penetracion_valor = f"{penetracion_producto[penetracion_producto['ZONA']==zona]['PENETRACION'].to_list()[0]}%"
#     cobertura_valor = f"{cobertura_hoy[cobertura_hoy['ZONA']==zona]['porcentaje_cobertura'].round(1).to_list()[0]}"
#     cobertura_valor_ayer = f"{cobertura_ayer[cobertura_ayer['ZONA']==zona]['porcentaje_cobertura'].round(1).to_list()[0]}"
#     clientes_valor = f"{cobertura_hoy[cobertura_hoy['ZONA']==zona]['sin compra'].round(1).to_list()[0]}"
#     clientes_valor_ayer = f"{cobertura_ayer[cobertura_ayer['ZONA']==zona]['sin compra'].round(1).to_list()[0]}"
#     penetracion_valor_clientes = f"{int(penetracion_producto[penetracion_producto['ZONA']==zona]['CLIENTE_sin_compra'].to_list()[0])}"
#     cumplimiento_hoy =   f"{cobertura_hoy[cobertura_hoy['ZONA']==zona]['cumplimiento%'].round(1).to_list()[0]}"
#     cumplimiento_ayer =   f"{cobertura_ayer[cobertura_ayer['ZONA']==zona]['cumplimiento%'].round(1).to_list()[0]}"
#     falta_hoy_valor =   float(f"{cobertura_hoy[cobertura_hoy['ZONA']==zona]['falta presu'].round(1).to_list()[0]}")
#     falta_ayer_valor =   float(f"{cobertura_ayer[cobertura_ayer['ZONA']==zona]['falta presu'].round(1).to_list()[0]}")
#     presupuesto= int(cobertura_hoy[cobertura_hoy['ZONA']==zona]['PRESUPUESTO'].to_list()[0])
#     ventas = int(cobertura_hoy[cobertura_hoy['ZONA']==zona]['ventas'].to_list()[0])
#     clientes_activos = int(cobertura_hoy[cobertura_hoy['ZONA']==zona]['total_clientes'].to_list()[0])
  
#     # --- GENERAR SECCIÓN DE TABLA TOP SOLO SI NO ES MAYORISTA ---
#     if zona != 'MAYORISTAS':
#         df_top = (
#             reporte_critico[reporte_critico['ZONA'] == zona][['Ranking', 'CLIENTE']]
#             .sort_values('Ranking')
#         )

#         # Generar filas HTML con estilos alternados
#         filas_html = ""
#         for idx, row in df_top.iterrows():
#             # Alternar colores de fondo
#             bg_color = "#f8f9fa" if row['Ranking'] % 2 == 0 else "#ffffff"
            
#             # Badge para el ranking
#             if row['Ranking'] <= 3:
#                 badge_color = "#e74c3c"  # Rojo para top 3
#                 badge_icon = "🔴"
#             elif row['Ranking'] <= 5:
#                 badge_color = "#f39c12"  # Naranja para 4-5
#                 badge_icon = "🟠"
#             else:
#                 badge_color = "#95a5a6"  # Gris para el resto
#                 badge_icon = "⚪"
            
#             filas_html += f"""
#             <tr style="background-color: {bg_color}; transition: background-color 0.2s;">
#                 <td style="padding: 14px 12px; border-bottom: 1px solid #e8e8e8; text-align: center; width: 80px;">
#                     <span style="background-color: {badge_color}; color: white; padding: 6px 12px; border-radius: 20px; font-weight: bold; font-size: 13px; display: inline-block; min-width: 35px;">
#                         {badge_icon} #{row['Ranking']}
#                     </span>
#                 </td>
#                 <td style="padding: 14px 16px; border-bottom: 1px solid #e8e8e8; color: #2c3e50; font-size: 14px; font-weight: 500;">
#                     {row['CLIENTE']}
#                 </td>
#             </tr>
#             """
        
#         # Construir tabla completa 
#         tabla_html = f"""
#         <div style="background: linear-gradient(to bottom, #ffffff, #f8f9fa); border-radius: 8px; overflow: hidden; box-shadow: 0 2px 8px rgba(0,0,0,0.08);">
#             <table width="100%" style="border-collapse: collapse; font-family: 'Segoe UI', Arial, sans-serif;">
#                 <thead>
#                     <tr style="background: linear-gradient(135deg, #2c3e50 0%, #34495e 100%);">
#                         <th style="padding: 16px 12px; color: #ffffff; text-align: center; font-size: 13px; font-weight: 600; text-transform: uppercase; letter-spacing: 0.5px; border-right: 1px solid rgba(255,255,255,0.1);">
#                             Ranking
#                         </th>
#                         <th style="padding: 16px; color: #ffffff; text-align: left; font-size: 13px; font-weight: 600; text-transform: uppercase; letter-spacing: 0.5px;">
#                             Cliente
#                         </th>
#                     </tr>
#                 </thead>
#                 <tbody>
#                     {filas_html}
#                 </tbody>
#             </table>
#         </div>
#         """
        
#         # Sección completa de clientes críticos
#         seccion_clientes_criticos = f"""
#             <tr>
#                 <td style="padding: 0 20px 30px 20px;">
#                     <div style="border-top: 1px solid #eee; padding-top: 20px;">
#                         <h3 style="color: #1a5276; font-size: 16px; margin-bottom: 15px; text-align: left; display: flex; align-items: center;">
#                             <span style="background-color: #e74c3c; color: white; padding: 4px 8px; border-radius: 4px; margin-right: 10px; font-size: 12px;">CRÍTICO</span>
#                             CLIENTES TOP SIN COMPRA
#                         </h3>
#                         {tabla_html}
#                     </div>
#                 </td>
#             </tr>
#         """
#     else:
#         seccion_clientes_criticos = ""  # No mostrar nada para MAYORISTA

#     # HTML DEL CORREO COMPLETO
#     cuerpo_correo = f"""
#     <!DOCTYPE html>
#     <html>
#     <head><meta charset="UTF-8"></head>
#     <body style="margin:0; padding:0; font-family: 'Segoe UI', Arial, sans-serif; background-color: #f4f7f9;">
#         <table align="center" border="0" cellpadding="0" cellspacing="0" width="600" style="border-collapse: collapse; background-color: #ffffff; margin-top: 20px; border: 1px solid #e0e0e0; border-radius: 8px; overflow: hidden;">
#             <tr>
#                 <td bgcolor="#1a5276" style="padding: 25px; text-align: center; color: #ffffff;">
#                     <h1 style="margin: 0; font-size: 20px; letter-spacing: 1px;">INFORME DIARIO DE VENTAS {hoy.month_name(locale='es_ES').upper()}</h1>
#                     <p style="margin: 5px 0 0 0; font-size: 13px; opacity: 0.9;">Resultado Acumulado al {ayer.date()} | <strong>Zona {zona}</strong></p>
#                 </td>
#             </tr>

#             <tr>
#                 <td style="padding: 20px;">
#                     <table width="100%" border="0" cellspacing="0" cellpadding="5">
#                         <tr>
#                             <td width="50%">
#                                 <div style="background-color: #f8f9fa; border-left: 4px solid #3498db; padding: 15px; border-radius: 4px;">
#                                     <span style="font-size: 11px; color: #7f8c8d; text-transform: uppercase; font-weight: bold;">Cobertura</span><br>
#                                     <span style="font-size: 18px; font-weight: bold; color: #2c3e50;">Hoy: {cobertura_valor}%</span><br>
#                                     <span style="font-size: 12px; color: #95a5a6;">{ayer.date()}: {cobertura_valor_ayer}%</span>
#                                 </div>
#                             </td>
#                             <td width="50%">
#                                 <div style="background-color: #f8f9fa; border-left: 4px solid #e67e22; padding: 15px; border-radius: 4px;">
#                                     <span style="font-size: 11px; color: #7f8c8d; text-transform: uppercase; font-weight: bold;">Activos sin Compra</span><br>
#                                     <span style="font-size: 18px; font-weight: bold; color: #2c3e50;">Hoy: {clientes_valor}</span><br>
#                                     <span style="font-size: 12px; color: #95a5a6;">{ayer.date()}: {clientes_valor_ayer}</span>
#                                 </div>
#                             </td>
#                         </tr>
#                         <tr><td colspan="2" style="height: 10px;"></td></tr>
#                         <tr>
#                             <td width="50%">
#                                 <div style="background-color: #ebf5fb; border-left: 4px solid #2ecc71; padding: 15px; border-radius: 4px;">
#                                     <span style="font-size: 11px; color: #7f8c8d; text-transform: uppercase; font-weight: bold;">Penetración {productos_analisis}</span><br>
#                                     <span style="font-size: 22px; font-weight: bold; color: #27ae60;">{penetracion_valor}</span>
#                                 </div>
#                             </td>
#                             <td width="50%">
#                                 <div style="background-color: #ebf5fb; border-left: 4px solid #2ecc71; padding: 15px; border-radius: 4px;">
#                                     <span style="font-size: 11px; color: #7f8c8d; text-transform: uppercase; font-weight: bold;">Sin Compra ({productos_analisis})</span><br>
#                                     <span style="font-size: 22px; font-weight: bold; color: #27ae60;">{penetracion_valor_clientes}/{clientes_activos} </span>
#                                 </div>
#                             </td>
#                         </tr>
#                         <tr><td colspan="2" style="height: 10px;"></td></tr>
#                         <tr>
#                             <td width="50%">
#                                 <div style="background-color: #f8f9fa; border-left: 4px solid #3498db; padding: 15px; border-radius: 4px;">
#                                     <span style="font-size: 11px; color: #7f8c8d; text-transform: uppercase; font-weight: bold;">Cumplimineto %</span><br>
#                                     <span style="font-size: 18px; font-weight: bold; color: #2c3e50;">Hoy: {cumplimiento_hoy}%</span><br>
#                                     <span style="font-size: 12px; color: #95a5a6;">{ayer.date()}: {cumplimiento_ayer}%</span>
#                                 </div>
#                             </td>
#                             <td width="50%">
#                                 <div style="background-color: #f8f9fa; border-left: 4px solid #e67e22; padding: 15px; border-radius: 4px;">
#                                     <span style="font-size: 11px; color: #7f8c8d; text-transform: uppercase; font-weight: bold;">Millones Faltantes</span><br>
#                                     <span style="font-size: 18px; font-weight: bold; color: #2c3e50;">Hoy: ${falta_hoy_valor:,.0f}</span><br>
#                                     <span style="font-size: 12px; color: #95a5a6;">{ayer.date()}: ${falta_ayer_valor:,.0f}</span>
#                                 </div>
#                             </td>
#                         </tr>
#                         <tr><td colspan="2" style="height: 10px;"></td></tr>
#                         <tr>
#                             <td width="50%">
#                                 <div style="background-color: #f8f9fa; border-left: 4px solid #3498db; padding: 15px; border-radius: 4px;">
#                                     <span style="font-size: 11px; color: #7f8c8d; text-transform: uppercase; font-weight: bold;">Ventas</span><br>
#                                     <span style="font-size: 18px; font-weight: bold; color: #2c3e50;">${ventas:,.0f}</span><br>
#                                 </div>
#                             </td>
#                             <td width="50%">
#                                 <div style="background-color: #f8f9fa; border-left: 4px solid #e67e22; padding: 15px; border-radius: 4px;">
#                                     <span style="font-size: 11px; color: #7f8c8d; text-transform: uppercase; font-weight: bold;">Presupuesto</span><br>
#                                     <span style="font-size: 18px; font-weight: bold; color: #2c3e50;">${presupuesto:,.0f}</span><br>
#                                 </div>
#                             </td>
#                         </tr>
#                     </table>
#                 </td>
#             </tr>
#             {seccion_clientes_criticos}
#             <tr>
#                 <td bgcolor="#f4f7f9" style="padding: 15px; text-align: center; font-size: 11px; color: #95a5a6; border-top: 1px solid #e0e0e0;">
#                     Este es un reporte automático generado por el área de Analisis de datos.
#                 </td>
#             </tr>
#         </table>

#     </body>
#     </html>
#     """
    
#     # 4. VISUALIZAR EN JUPYTER
#     print(f"--- Previsualización para: {zona} ---")
#     display(HTML(cuerpo_correo))
    
#     # Opcional: romper el bucle tras la primera zona
#     # break

In [9]:
from IPython.display import display, HTML

for i in responsables:

    filas_html = ""
    df = cobertura_clientes[cobertura_clientes['RESPONSABLE'] == i]

    for _, row in df.iterrows():

        cumpl_color = (
            "#d5f5e3" if row['cumplimiento%'] >= 100
            else "#fcf3cf" if row['cumplimiento%'] >= 80
            else "#fadbd8"
        )

        falta_color = "#c0392b" if row['falta presu'] > 0 else "#1e8449"

        filas_html += f"""
        <tr style="transition: background-color 0.2s;">
            <td style="padding:12px 16px;border-bottom:1px solid #e8e8e8;
                       font-weight:500;color:#2c3e50;font-size:13px;">
                {row['CLIENTE']}
            </td>

            <td style="padding:12px 16px;border-bottom:1px solid #e8e8e8;
                       text-align:right;font-weight:600;color:#34495e;font-size:13px;">
                ${row['PRESUPUESTO']:,.0f}
            </td>

            <td style="padding:12px 16px;border-bottom:1px solid #e8e8e8;
                       text-align:right;font-weight:600;color:#34495e;font-size:13px;">
                ${row['ventas']:,.0f}
            </td>

            <td style="padding:10px 14px;border-bottom:1px solid #e8e8e8;
                       text-align:center;font-weight:700;
                       background-color:{cumpl_color};
                       color:#2c3e50;font-size:13px;
                       border-radius:4px;">
                {row['cumplimiento%']:.1f}%
            </td>

            <td style="padding:12px 16px;border-bottom:1px solid #e8e8e8;
                       text-align:right;font-weight:700;
                       color:{falta_color};font-size:13px;">
                ${row['falta presu']:,.0f}
            </td>

            <td style="padding:12px 16px;border-bottom:1px solid #e8e8e8;
                       text-align:center;font-weight:600;color:#34495e;font-size:13px;">
                {int(row['productos_comprados'])}/{int(row['productos_pocion'])}
            </td>
        </tr>
        """

    tabla_html = f"""
    <!DOCTYPE html>
    <html>
    <head>
        <meta charset="UTF-8">
    </head>

    <body style="margin:0;padding:20px 0;background-color:#f4f6f7;font-family:'Segoe UI', -apple-system, system-ui, sans-serif;">

    <div style="
        max-width:900px;
        margin:0 auto;
        background:#ffffff;
        border-radius:8px;
        overflow:hidden;
        box-shadow: 0 2px 8px rgba(0,0,0,0.08);
    ">

        <table width="100%" cellpadding="0" cellspacing="0" style="border-collapse:collapse;">

            <thead>
                <tr>
                    <td colspan="6" bgcolor="#1a5276" style="padding: 25px; text-align: center; color: #ffffff;">
                        <h1 style="margin: 0; font-size: 20px; letter-spacing: 1px;">INFORME DIARIO DE VENTAS {hoy.month_name(locale='es_ES').upper()}</h1>
                        <p style="margin: 5px 0 0 0; font-size: 13px; opacity: 0.9;">Resultado Acumulado al {ayer.date()} | <strong>Zona {i}</strong></p>
                    </td>
                </tr>
                <tr style="background: linear-gradient(135deg, #1a5276 0%, #2471a3 100%);">
                    <th style="padding:16px;color:#ffffff;text-align:left;font-size:12px;font-weight:600;text-transform:uppercase;letter-spacing:0.5px;width:28%;">
                        Cliente
                    </th>
                    <th style="padding:16px;color:#ffffff;text-align:right;font-size:12px;font-weight:600;text-transform:uppercase;letter-spacing:0.5px;width:16%;">
                        Presupuesto
                    </th>
                    <th style="padding:16px;color:#ffffff;text-align:right;font-size:12px;font-weight:600;text-transform:uppercase;letter-spacing:0.5px;width:16%;">
                        Ventas
                    </th>
                    <th style="padding:16px;color:#ffffff;text-align:center;font-size:12px;font-weight:600;text-transform:uppercase;letter-spacing:0.5px;width:13%;">
                        Cumpl. %
                    </th>
                    <th style="padding:16px;color:#ffffff;text-align:right;font-size:12px;font-weight:600;text-transform:uppercase;letter-spacing:0.5px;width:15%;">
                        Falta $
                    </th>
                    <th style="padding:16px;color:#ffffff;text-align:center;font-size:12px;font-weight:600;text-transform:uppercase;letter-spacing:0.5px;width:12%;">
                        Penetración
                    </th>
                </tr>
            </thead>

            <tbody style="background-color:#ffffff;">
                {filas_html}
            </tbody>
        </table>

        <div style="background-color:#f8f9fa;padding:16px;text-align:center;border-top:1px solid #e0e0e0;">
            <p style="margin:0;font-size:11px;color:#7f8c8d;line-height:1.6;">
                Reporte automático generado por el Área de Análisis de Datos
            </p>
        </div>
    </div>

    </body>
    </html>
    """

    print(f"--- Previsualización para: {i} ---")
    display(HTML(tabla_html))

--- Previsualización para: DANIELA ANGULO ---


--- Previsualización para: DIANA RIOS ---
